# Calendar & Promotion Feature Engineering

## Business Objective

Evaluate whether calendar characteristics and historical promotional activity provide useful information for forecasting total monthly retail sales.

## Forecasting Design

- **Target:** Monthly Total Sales
- **Historical period:** January 2013–July 2017
- **Forecast horizon:** 3 months
- **Minimum training history:** 36 months
- **Validation:** 17 expanding rolling-origin windows
- **Forecast cases:** 51
- **Existing benchmark:** 50/50 Blended model — 8.36% WAPE
- **Alternative benchmark:** ETS Additive — 8.48% WAPE

## Data Governance Rules

1. Do not modify raw or standardized source datasets.
2. Preserve the monthly forecasting grain.
3. Use only information available at the relevant forecast origin.
4. Do not use realized future promotions as predictors.
5. Validate features before modeling.
6. Compare models on identical historical forecast cases.

## Step 1 — Feature Availability Audit

We will first inspect the existing analytical datasets and identify suitable calendar and promotion variables.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = next(
    parent
    for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / "data" / "processed").exists()
)

print("Project root:", PROJECT_ROOT)
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)

required_files = [
    "data/processed/monthly_sales_complete.csv",
    "data/processed/train_standardized.csv",
    "reports/retail_driver_analysis/monthly_sales_promotion_performance.csv",
    "reports/forecasting_feature_evaluation/seven_model_matched_forecasts.csv",
]

for relative_path in required_files:
    file_path = PROJECT_ROOT / relative_path
    print(
        "FOUND" if file_path.is_file() else "MISSING",
        relative_path
    )

assert all(
    (PROJECT_ROOT / path).is_file()
    for path in required_files
)

print("Environment and input file validation: PASSED")

Project root: d:\GitHub\retail-sales-forecasting
Pandas: 3.0.6
NumPy: 2.5.3
FOUND data/processed/monthly_sales_complete.csv
FOUND data/processed/train_standardized.csv
FOUND reports/retail_driver_analysis/monthly_sales_promotion_performance.csv
FOUND reports/forecasting_feature_evaluation/seven_model_matched_forecasts.csv
Environment and input file validation: PASSED


In [2]:
# Load existing analytical datasets

monthly_sales_df = pd.read_csv(
    PROJECT_ROOT / "data/processed/monthly_sales_complete.csv"
)

monthly_promotion_df = pd.read_csv(
    PROJECT_ROOT
    / "reports/retail_driver_analysis/monthly_sales_promotion_performance.csv"
)

# Inspect dataset structure
print("MONTHLY SALES DATASET")
print("Rows:", len(monthly_sales_df))
print("Columns:", monthly_sales_df.columns.tolist())
print("\nData types:")
print(monthly_sales_df.dtypes)

print("\n" + "=" * 50)

print("MONTHLY PROMOTION DATASET")
print("Rows:", len(monthly_promotion_df))
print("Columns:", monthly_promotion_df.columns.tolist())
print("\nData types:")
print(monthly_promotion_df.dtypes)

MONTHLY SALES DATASET
Rows: 55
Columns: ['month', 'total_sales', 'source_rows', 'sales_dates_observed', 'stores_observed', 'families_observed', 'calendar_days', 'sales_closure_days', 'is_partial_month', 'sales_coverage_complete', 'is_forecast_training_period']

Data types:
month                              str
total_sales                    float64
source_rows                      int64
sales_dates_observed             int64
stores_observed                  int64
families_observed                int64
calendar_days                    int64
sales_closure_days               int64
is_partial_month                  bool
sales_coverage_complete           bool
is_forecast_training_period       bool
dtype: object

MONTHLY PROMOTION DATASET
Rows: 55
Columns: ['month', 'monthly_sales', 'total_records', 'promotional_records', 'total_promoted_item_counts', 'promotion_record_share_pct', 'avg_sales_per_record']

Data types:
month                             str
monthly_sales                 float6

In [3]:
# Convert month columns to datetime

monthly_sales_df["month"] = pd.to_datetime(
    monthly_sales_df["month"]
)

monthly_promotion_df["month"] = pd.to_datetime(
    monthly_promotion_df["month"]
)

# Validate monthly keys
assert monthly_sales_df["month"].notna().all()
assert monthly_promotion_df["month"].notna().all()

assert monthly_sales_df["month"].is_unique
assert monthly_promotion_df["month"].is_unique

assert monthly_sales_df["month"].dt.is_month_start.all()
assert monthly_promotion_df["month"].dt.is_month_start.all()

# Compare month coverage
sales_months = set(monthly_sales_df["month"])
promotion_months = set(monthly_promotion_df["month"])

assert sales_months == promotion_months

# Reconcile monthly sales values
reconciliation_df = monthly_sales_df[
    ["month", "total_sales"]
].merge(
    monthly_promotion_df[
        ["month", "monthly_sales"]
    ],
    on="month",
    how="outer",
    validate="one_to_one",
    indicator=True
)

assert (reconciliation_df["_merge"] == "both").all()

reconciliation_df["sales_difference"] = (
    reconciliation_df["total_sales"]
    - reconciliation_df["monthly_sales"]
)

max_difference = (
    reconciliation_df["sales_difference"].abs().max()
)

assert max_difference < 0.01

print("Monthly key validation: PASSED")
print("Sales reconciliation: PASSED")
print("Matched months:", len(reconciliation_df))
print("Maximum sales difference:", round(max_difference, 6))
print(
    "Date range:",
    reconciliation_df["month"].min().date(),
    "to",
    reconciliation_df["month"].max().date()
)

Monthly key validation: PASSED
Sales reconciliation: PASSED
Matched months: 55
Maximum sales difference: 0.0
Date range: 2013-01-01 to 2017-07-01


In [4]:
# Select the relevant promotion columns
promotion_features = monthly_promotion_df[
    [
        "month",
        "total_records",
        "promotional_records",
        "total_promoted_item_counts",
        "promotion_record_share_pct",
    ]
].copy()

# Merge with monthly sales
monthly_features_df = monthly_sales_df.merge(
    promotion_features,
    on="month",
    how="left",
    validate="one_to_one",
    indicator=True
)

# Validate the merge
assert len(monthly_features_df) == 55
assert (monthly_features_df["_merge"] == "both").all()
assert monthly_features_df["month"].is_unique

monthly_features_df = (
    monthly_features_df
    .drop(columns="_merge")
    .sort_values("month")
    .reset_index(drop=True)
)

# Check missing values in selected columns
selected_columns = [
    "month",
    "total_sales",
    "calendar_days",
    "total_records",
    "promotional_records",
    "total_promoted_item_counts",
    "promotion_record_share_pct",
]

assert not monthly_features_df[
    selected_columns
].isna().any().any()

print("Monthly feature merge validation: PASSED")
print("Rows:", len(monthly_features_df))
print("Columns:", len(monthly_features_df.columns))

display(monthly_features_df[selected_columns].head())

Monthly feature merge validation: PASSED
Rows: 55
Columns: 15


,month,total_sales,calendar_days,total_records,promotional_records,total_promoted_item_counts,promotion_record_share_pct
0,2013-01-01,1.032762e+07,31,55242,0,0,0.0
1,2013-02-01,9.658960e+06,28,49896,0,0,0.0
2,2013-03-01,1.142850e+07,31,55242,0,0,0.0
3,2013-04-01,1.099346e+07,30,53460,0,0,0.0
4,2013-05-01,1.159770e+07,31,55242,0,0,0.0


## Step 2 — Forecast-Time Feature Availability Audit

### Forecasting scenario

At the end of April 2017, predict monthly sales for May, June, and July 2017.

### Known-in-advance features

- Calendar month
- Calendar year
- Days in month
- Calendar quarter

### Historical-only features

- Previous observed monthly sales
- Historical promotional record share
- Historical promoted-item counts
- Historical store coverage

Historical values must be restricted to observations available at the forecast origin.

### Features requiring caution

- `sales_closure_days`: May describe observed closures rather than advance-known closures.
- `stores_observed`: Historical operational coverage, not a guaranteed future store count.
- `total_records`: Actual future transaction/source-record counts are unknown.
- `promotional_records`: Actual future promotion counts are unknown.
- `promotion_record_share_pct`: Actual future promotion share is unknown.
- `total_promoted_item_counts`: Actual future promoted-item counts are unknown.

### Data leakage prevention

Do not use realized values from May, June, or July 2017 when generating forecasts from an April 2017 origin.

### Promotion data quality question

Investigate the first and last months with recorded promotions. A zero may represent no promotions or a source-data limitation; the current dataset alone does not establish which explanation is correct.

In [5]:
# Investigate monthly promotion coverage

promotion_audit_df = monthly_features_df[
    [
        "month",
        "total_records",
        "promotional_records",
        "total_promoted_item_counts",
        "promotion_record_share_pct",
    ]
].copy()

promotion_audit_df["has_recorded_promotions"] = (
    promotion_audit_df["promotional_records"] > 0
)

positive_months = promotion_audit_df[
    promotion_audit_df["has_recorded_promotions"]
]

zero_months = promotion_audit_df[
    ~promotion_audit_df["has_recorded_promotions"]
]

assert (
    promotion_audit_df["promotional_records"]
    <= promotion_audit_df["total_records"]
).all()

assert (
    promotion_audit_df[
        ["total_records", "promotional_records",
         "total_promoted_item_counts"]
    ] >= 0
).all().all()

assert promotion_audit_df[
    "promotion_record_share_pct"
].between(0, 100).all()

print("Promotion coverage audit: PASSED")
print("Total months:", len(promotion_audit_df))
print("Months with recorded promotions:", len(positive_months))
print("Months without recorded promotions:", len(zero_months))

if not positive_months.empty:
    print(
        "First recorded promotion month:",
        positive_months["month"].min().date()
    )
    print(
        "Last recorded promotion month:",
        positive_months["month"].max().date()
    )

print("\nFirst 15 months:")
display(promotion_audit_df.head(15))

Promotion coverage audit: PASSED
Total months: 55
Months with recorded promotions: 40
Months without recorded promotions: 15
First recorded promotion month: 2014-04-01
Last recorded promotion month: 2017-07-01

First 15 months:


,month,total_records,promotional_records,total_promoted_item_counts,promotion_record_share_pct,has_recorded_promotions
0,2013-01-01,55242,0,0,0.0,False
1,2013-02-01,49896,0,0,0.0,False
2,2013-03-01,55242,0,0,0.0,False
3,2013-04-01,53460,0,0,0.0,False
4,2013-05-01,55242,0,0,0.0,False
5,2013-06-01,53460,0,0,0.0,False
6,2013-07-01,55242,0,0,0.0,False
7,2013-08-01,55242,0,0,0.0,False
8,2013-09-01,53460,0,0,0.0,False
9,2013-10-01,55242,0,0,0.0,False


In [6]:
# Identify the first month with recorded promotions
first_promotion_month = promotion_audit_df.loc[
    promotion_audit_df["has_recorded_promotions"],
    "month"
].min()

# Create a conservative promotion coverage flag
monthly_features_df["promotion_coverage_status"] = np.where(
    monthly_features_df["month"] < first_promotion_month,
    "COVERAGE_UNCERTAIN",
    "RECORDED_PERIOD"
)

# Validation
assert first_promotion_month == pd.Timestamp("2014-04-01")

assert (
    monthly_features_df["promotion_coverage_status"]
    .value_counts()
    .to_dict()
    == {
        "RECORDED_PERIOD": 40,
        "COVERAGE_UNCERTAIN": 15,
    }
)

assert monthly_features_df["month"].is_unique
assert len(monthly_features_df) == 55

print("Promotion coverage classification: PASSED")
print("First recorded promotion:", first_promotion_month.date())

print("\nCoverage classification:")
print(
    monthly_features_df[
        "promotion_coverage_status"
    ].value_counts()
)

display(
    monthly_features_df.loc[
        monthly_features_df["month"].between(
            "2014-02-01", "2014-06-01"
        ),
        [
            "month",
            "promotional_records",
            "promotion_record_share_pct",
            "promotion_coverage_status",
        ]
    ]
)

Promotion coverage classification: PASSED
First recorded promotion: 2014-04-01

Coverage classification:
promotion_coverage_status
RECORDED_PERIOD       40
COVERAGE_UNCERTAIN    15
Name: count, dtype: int64


,month,promotional_records,promotion_record_share_pct,promotion_coverage_status
13,2014-02-01,0,0.000000,COVERAGE_UNCERTAIN
14,2014-03-01,0,0.000000,COVERAGE_UNCERTAIN
15,2014-04-01,399,0.746352,RECORDED_PERIOD
16,2014-05-01,1964,3.555266,RECORDED_PERIOD
17,2014-06-01,4367,8.168724,RECORDED_PERIOD


## Step 3 — Forecast-Time Feature Availability Decisions

The feature audit separates information known in advance,
historical information available at the forecast origin,
and realized information that cannot be used for future forecasts.

Promotion coverage is uncertain before April 2014.

Historical promotion variables will require origin-aware
feature engineering. No future realized promotion measures
will be used as predictors.

In [7]:
feature_audit_records = [
    ("month", "KNOWN_IN_ADVANCE", "Calendar month is known"),
    ("calendar_days", "KNOWN_IN_ADVANCE", "Calendar days are known"),
    ("total_sales", "TARGET", "Future sales must not be used"),
    ("promotion_record_share_pct", "HISTORICAL_ONLY",
     "Use only observations available at forecast origin"),
    ("total_promoted_item_counts", "HISTORICAL_ONLY",
     "Use only observations available at forecast origin"),
    ("promotional_records", "HISTORICAL_ONLY",
     "Future realized counts are unavailable"),
    ("sales_closure_days", "REQUIRES_INVESTIGATION",
     "Confirm whether closure dates were known in advance"),
    ("stores_observed", "HISTORICAL_ONLY",
     "Future observed store coverage is unknown"),
    ("total_records", "HISTORICAL_ONLY",
     "Future realized source-record counts are unknown"),
    ("promotion_coverage_status", "QUALITY_FLAG",
     "Early promotion coverage is uncertain"),
]

feature_audit_df = pd.DataFrame(
    feature_audit_records,
    columns=["feature", "availability", "reason"]
)

assert feature_audit_df["feature"].is_unique
assert feature_audit_df["feature"].isin(
    monthly_features_df.columns
).all()

assert len(feature_audit_df) == 10

print("Feature availability audit validation: PASSED")
print("Features audited:", len(feature_audit_df))

display(feature_audit_df)

Feature availability audit validation: PASSED
Features audited: 10


,feature,availability,reason
0,month,KNOWN_IN_ADVANCE,Calendar month is known
1,calendar_days,KNOWN_IN_ADVANCE,Calendar days are known
2,total_sales,TARGET,Future sales must not be used
3,promotion_record_share_pct,HISTORICAL_ONLY,Use only observations available at forecast or...
4,total_promoted_item_counts,HISTORICAL_ONLY,Use only observations available at forecast or...
5,promotional_records,HISTORICAL_ONLY,Future realized counts are unavailable
6,sales_closure_days,REQUIRES_INVESTIGATION,Confirm whether closure dates were known in ad...
7,stores_observed,HISTORICAL_ONLY,Future observed store coverage is unknown
8,total_records,HISTORICAL_ONLY,Future realized source-record counts are unknown
9,promotion_coverage_status,QUALITY_FLAG,Early promotion coverage is uncertain


In [8]:
# Create the report directory
report_dir = (
    PROJECT_ROOT
    / "reports"
    / "forecasting_feature_engineering"
)

report_dir.mkdir(parents=True, exist_ok=True)

# Export feature availability decisions
audit_path = report_dir / "feature_availability_audit.csv"

feature_audit_df.to_csv(
    audit_path,
    index=False
)

# Read the report back for validation
saved_audit_df = pd.read_csv(audit_path)

assert len(saved_audit_df) == 10
assert saved_audit_df["feature"].is_unique

pd.testing.assert_frame_equal(
    feature_audit_df.reset_index(drop=True),
    saved_audit_df.reset_index(drop=True)
)

print("Feature availability audit export: PASSED")
print("Report:", audit_path.relative_to(PROJECT_ROOT))
print("Features exported:", len(saved_audit_df))

Feature availability audit export: PASSED
Report: reports\forecasting_feature_engineering\feature_availability_audit.csv
Features exported: 10
